In [ ]:
from model import TransformerModel as Model
from tokenizer import SubwordTokenizer
import torch
from tqdm import tqdm

torch.set_default_device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
max_iters = 1000
eval_interval = 250
eval_iters = 200
window_size = 512
batch_size = 64
n_embed = 384
learning_rate = 1e-4
num_heads = 8
num_transformer_blocks = 8
dropout = 0.2

In [ ]:
text_path = ""
with open(text_path, 'r', encoding='utf-8') as f:
    text = f.read()
print("text length: ",len(text))
text[:1000]

In [ ]:
# tokenizer_train_text_path = ""
# with open(tokenizer_train_text_path, 'r', encoding='utf-8') as f:
#     text_to_train_tokenizer = f.read()

text_to_train_tokenizer = text

In [ ]:
tokenizer = SubwordTokenizer()
# configs_path = ""
# tokenizer.load_configs(configs_path)

In [ ]:
tokenizer.fit(text_to_train_tokenizer)

In [ ]:
path_to_save_configs = ""
tokenizer.save_configs()

In [ ]:
vocab_size = tokenizer.get_vocab_size()

In [ ]:
text = tokenizer.normalize(text)
decoded = tokenizer.encode(text)
data = torch.tensor(decoded)

In [ ]:
path_to_save_data = ""
torch.save(data, f'{path_to_save_data}/tiny_stories_data.pt')

In [ ]:
data_path = ""
data = torch.load(data_path)

In [ ]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    # takes a random batch of input x and target y from data
    # this function is used multiple times, it's not only-one-time-use
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - window_size, (batch_size,))
    x = torch.stack([data[i:i + window_size] for i in ix])
    y = torch.stack([data[i + 1:i + window_size + 1] for i in ix])
    return x ,y

In [ ]:
model = Model(n_embed, window_size, vocab_size, num_heads, num_transformer_blocks, dropout)

In [ ]:
@torch.no_grad()
def estimate_loss():
    # this function estimates loss while training for both train and eval dataset
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in tqdm(range(eval_iters)):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
print(sum(p.numel() for p in model.parameters()) / 1e6, 'M parameters')

In [ ]:
for iter in tqdm(range(max_iters)):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f'step {iter}: train_loss {losses['train']:.4f}, val_loss {losses['val']:.4f}')

    xb, yb = get_batch('train_data')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # print(loss.item())

In [ ]:
# input_text = "A"
inp_d = torch.zeros((1, 1), dtype=torch.long) # or torch.tensor( tokenizer.encode( input_text ) , dtype=torch.long).reshape(1,-1 )

In [ ]:
out = model.generate( inp_d , max_new_tokens=1000)[0].tolist()

In [ ]:
print(tokenizer.decode(out))

In [ ]:
save_path = ""
model_name = ""


torch.save(model, f'{save_path}/{model_name}.pth')